In [1]:
# ============================================================
# HUNGARIAN TEXT CLUSTERING USING NLP + K-MEANS
# Dataset: hungarian.csv
# ============================================================


# ============================================================
# 1. INSTALL LIBRARIES
# ============================================================

# Run this cell if packages are not installed.
# In Jupyter Notebook, uncomment the line below.

# !pip install pandas numpy matplotlib seaborn scikit-learn nltk


# ============================================================
# 2. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import TruncatedSVD

import re
import warnings

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")


# ============================================================
# 3. LOAD DATASET
# ============================================================

file_path = "hungarian.csv"

df = pd.read_csv(
    file_path,
    encoding="utf-8"
)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())


# ============================================================
# 4. BASIC DATASET INFORMATION
# ============================================================

print("\n" + "=" * 60)
print("DATASET INFORMATION")
print("=" * 60)

print("\nNumber of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())


# ============================================================
# 5. REMOVE DUPLICATES
# ============================================================

df = df.drop_duplicates().reset_index(drop=True)

print("\nDataset after removing duplicates:")
print(df.shape)


# ============================================================
# 6. FIND TEXT COLUMN
# ============================================================

# Find columns containing strings
text_columns = []

for column in df.columns:
    if df[column].dtype == "object":
        text_columns.append(column)

print("\nPossible text columns:")
for column in text_columns:
    print("-", column)


# ------------------------------------------------------------
# IMPORTANT:
#
# If the automatic selection chooses the wrong column,
# change the following line manually.
#
# Example:
# text_column = "text"
# text_column = "article"
# text_column = "sentence"
# ------------------------------------------------------------

if len(text_columns) == 0:
    raise ValueError(
        "No text column was found in the CSV file."
    )

text_column = text_columns[0]

print("\nSelected text column:", text_column)


# ============================================================
# 7. HANDLE MISSING TEXT
# ============================================================

df[text_column] = df[text_column].fillna("")

df[text_column] = df[text_column].astype(str)

# Remove completely empty documents
df = df[df[text_column].str.strip() != ""]

df = df.reset_index(drop=True)

print("\nNumber of usable documents:", len(df))


# ============================================================
# 8. HUNGARIAN STOPWORDS
# ============================================================

hungarian_stopwords = {
    "a",
    "az",
    "egy",
    "és",
    "is",
    "hogy",
    "nem",
    "de",
    "ha",
    "van",
    "volt",
    "lesz",
    "meg",
    "már",
    "mint",
    "ezt",
    "azt",
    "ez",
    "itt",
    "ott",
    "ami",
    "aki",
    "akik",
    "amely",
    "amelyek",
    "mert",
    "vagy",
    "vagyis",
    "csak",
    "még",
    "igen",
    "sok",
    "más",
    "minden",
    "mind",
    "nagy",
    "úgy",
    "így",
    "majd",
    "kell",
    "lehet",
    "kellett",
    "szerint",
    "között",
    "után",
    "előtt",
    "alatt",
    "fölött",
    "felett",
    "felé",
    "által",
    "mi",
    "te",
    "ő",
    "miért",
    "hogyan",
    "mikor",
    "hol",
    "aki",
    "akik",
    "amely",
    "amelyek",
    "azon",
    "azonban",
    "arra",
    "arról",
    "azért",
    "ebben",
    "ebből",
    "eddig",
    "ennek",
    "ennek",
    "ilyen",
    "ilyesmi",
    "jól",
    "kell",
    "kívül",
    "le",
    "ma",
    "most",
    "nagyon",
    "ne",
    "nélkül",
    "oda",
    "onnan",
    "ott",
    "pedig",
    "rá",
    "saját",
    "sem",
    "soha",
    "több",
    "tovább",
    "től",
    "tőle",
    "valaki",
    "valami",
    "valamint",
    "vannak"
}

print("Number of Hungarian stopwords:", len(hungarian_stopwords))


# ============================================================
# 9. TEXT CLEANING FUNCTION
# ============================================================

def clean_hungarian_text(text):
    """
    Clean Hungarian text.

    Operations:
    1. Convert to lowercase
    2. Remove URLs
    3. Remove email addresses
    4. Remove numbers
    5. Keep Hungarian characters
    6. Remove punctuation
    7. Remove extra whitespace
    """

    text = str(text)

    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(
        r"https?://\S+|www\.\S+",
        " ",
        text
    )

    # Remove email addresses
    text = re.sub(
        r"\S+@\S+",
        " ",
        text
    )

    # Keep Hungarian letters and spaces
    text = re.sub(
        r"[^a-záéíóöőúüű\s]",
        " ",
        text
    )

    # Remove extra whitespace
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


# ============================================================
# 10. APPLY TEXT CLEANING
# ============================================================

df["clean_text"] = df[text_column].apply(
    clean_hungarian_text
)

print("\nOriginal text:")
print(df[text_column].iloc[0])

print("\nCleaned text:")
print(df["clean_text"].iloc[0])


# ============================================================
# 11. REMOVE HUNGARIAN STOPWORDS
# ============================================================

def remove_stopwords(text):
    words = text.split()

    filtered_words = [
        word
        for word in words
        if word not in hungarian_stopwords
    ]

    return " ".join(filtered_words)


df["processed_text"] = df["clean_text"].apply(
    remove_stopwords
)

print("\nProcessed text:")
print(df["processed_text"].iloc[0])


# ============================================================
# 12. REMOVE VERY SHORT DOCUMENTS
# ============================================================

df = df[
    df["processed_text"].str.split().str.len() >= 2
]

df = df.reset_index(drop=True)

print("\nDocuments after preprocessing:", len(df))


# ============================================================
# 13. TF-IDF VECTORIZATION
# ============================================================

vectorizer = TfidfVectorizer(
    lowercase=False,
    max_features=10000,
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 2),
    sublinear_tf=True
)

X = vectorizer.fit_transform(
    df["processed_text"]
)

print("\n" + "=" * 60)
print("TF-IDF")
print("=" * 60)

print("TF-IDF matrix shape:", X.shape)
print("Number of documents:", X.shape[0])
print("Number of features:", X.shape[1])


# ============================================================
# 14. SHOW TF-IDF FEATURES
# ============================================================

feature_names = vectorizer.get_feature_names_out()

print("\nFirst 50 TF-IDF features:")
print(feature_names[:50])


# ============================================================
# 15. FIND TOP GLOBAL TF-IDF WORDS
# ============================================================

mean_tfidf = np.asarray(
    X.mean(axis=0)
).flatten()

top_indices = mean_tfidf.argsort()[::-1][:30]

print("\n" + "=" * 60)
print("TOP TF-IDF FEATURES")
print("=" * 60)

for rank, index in enumerate(top_indices, start=1):
    print(
        f"{rank:2d}. "
        f"{feature_names[index]:30s} "
        f"{mean_tfidf[index]:.4f}"
    )


# ============================================================
# 16. FIND OPTIMAL NUMBER OF CLUSTERS
# ============================================================

# We test K from 2 to 10.
#
# If your dataset is very small, reduce this range.

k_values = range(2, 11)

inertias = []
silhouette_scores = []

print("\n" + "=" * 60)
print("TESTING DIFFERENT K VALUES")
print("=" * 60)

for k in k_values:

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = model.fit_predict(X)

    inertia = model.inertia_

    silhouette = silhouette_score(
        X,
        labels
    )

    inertias.append(inertia)
    silhouette_scores.append(silhouette)

    print(
        f"K = {k:2d} | "
        f"Inertia = {inertia:.4f} | "
        f"Silhouette = {silhouette:.4f}"
    )


# ============================================================
# 17. ELBOW METHOD GRAPH
# ============================================================

plt.figure(figsize=(10, 6))

plt.plot(
    list(k_values),
    inertias,
    marker="o"
)

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method for K-Means")

plt.xticks(list(k_values))
plt.grid(True)

plt.show()


# ============================================================
# 18. SILHOUETTE SCORE GRAPH
# ============================================================

plt.figure(figsize=(10, 6))

plt.plot(
    list(k_values),
    silhouette_scores,
    marker="o"
)

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score for K-Means")

plt.xticks(list(k_values))
plt.grid(True)

plt.show()


# ============================================================
# 19. SELECT BEST K
# ============================================================

best_index = np.argmax(
    silhouette_scores
)

best_k = list(k_values)[best_index]

best_score = silhouette_scores[best_index]

print("\n" + "=" * 60)
print("BEST K")
print("=" * 60)

print("Best number of clusters:", best_k)
print("Best silhouette score:", round(best_score, 4))


# ============================================================
# 20. TRAIN FINAL K-MEANS MODEL
# ============================================================

kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=20,
    max_iter=300
)

cluster_labels = kmeans.fit_predict(X)

df["cluster"] = cluster_labels

print("\nK-Means clustering completed.")


# ============================================================
# 21. CLUSTER COUNTS
# ============================================================

cluster_counts = (
    df["cluster"]
    .value_counts()
    .sort_index()
)

print("\n" + "=" * 60)
print("DOCUMENTS PER CLUSTER")
print("=" * 60)

print(cluster_counts)


# ============================================================
# 22. CLUSTER DISTRIBUTION GRAPH
# ============================================================

plt.figure(figsize=(10, 6))

sns.countplot(
    data=df,
    x="cluster"
)

plt.xlabel("Cluster")
plt.ylabel("Number of Documents")
plt.title("Number of Documents in Each Cluster")

plt.show()


# ============================================================
# 23. TOP WORDS FOR EACH CLUSTER
# ============================================================

print("\n" + "=" * 70)
print("TOP WORDS FOR EACH CLUSTER")
print("=" * 70)

order_centroids = (
    kmeans.cluster_centers_
    .argsort(axis=1)[:, ::-1]
)

cluster_keywords = {}

for cluster_number in range(best_k):

    top_terms = [
        feature_names[index]
        for index in order_centroids[
            cluster_number, :20
        ]
    ]

    cluster_keywords[cluster_number] = top_terms

    print(
        f"\nCluster {cluster_number}:"
    )

    print(
        ", ".join(top_terms)
    )


# ============================================================
# 24. DISPLAY SAMPLE DOCUMENTS FROM EACH CLUSTER
# ============================================================

print("\n" + "=" * 70)
print("SAMPLE DOCUMENTS FROM EACH CLUSTER")
print("=" * 70)

for cluster_number in range(best_k):

    print("\n")
    print("#" * 70)
    print(f"CLUSTER {cluster_number}")
    print("#" * 70)

    cluster_data = df[
        df["cluster"] == cluster_number
    ]

    number_of_examples = min(
        5,
        len(cluster_data)
    )

    for i in range(number_of_examples):

        text = cluster_data[
            text_column
        ].iloc[i]

        print(
            f"\nExample {i + 1}:"
        )

        print(
            str(text)[:1000]
        )


# ============================================================
# 25. REDUCE TF-IDF TO 2 DIMENSIONS
# ============================================================

svd = TruncatedSVD(
    n_components=2,
    random_state=42
)

X_2d = svd.fit_transform(X)

df["component_1"] = X_2d[:, 0]
df["component_2"] = X_2d[:, 1]

print("\n2D dimensionality reduction completed.")


# ============================================================
# 26. VISUALIZE K-MEANS CLUSTERS
# ============================================================

plt.figure(figsize=(12, 8))

sns.scatterplot(
    data=df,
    x="component_1",
    y="component_2",
    hue="cluster",
    palette="tab10",
    s=70,
    alpha=0.8
)

plt.title(
    "K-Means Clustering of Hungarian Text"
)

plt.xlabel("SVD Component 1")
plt.ylabel("SVD Component 2")

plt.legend(
    title="Cluster",
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

plt.grid(True)

plt.tight_layout()

plt.show()


# ============================================================
# 27. FINAL SILHOUETTE SCORE
# ============================================================

final_silhouette = silhouette_score(
    X,
    df["cluster"]
)

print("\n" + "=" * 60)
print("FINAL MODEL EVALUATION")
print("=" * 60)

print(
    "Silhouette Score:",
    round(final_silhouette, 4)
)


# ============================================================
# 28. CREATE CLUSTER SUMMARY TABLE
# ============================================================

summary_data = []

for cluster_number in range(best_k):

    cluster_size = len(
        df[df["cluster"] == cluster_number]
    )

    keywords = ", ".join(
        cluster_keywords[cluster_number][:10]
    )

    summary_data.append({
        "Cluster": cluster_number,
        "Documents": cluster_size,
        "Top Keywords": keywords
    })

cluster_summary = pd.DataFrame(
    summary_data
)

print("\n" + "=" * 60)
print("CLUSTER SUMMARY")
print("=" * 60)

display(cluster_summary)


# ============================================================
# 29. ADD CLUSTER KEYWORDS TO EACH DOCUMENT
# ============================================================

df["cluster_keywords"] = df["cluster"].apply(
    lambda cluster:
        ", ".join(
            cluster_keywords[cluster][:10]
        )
)


# ============================================================
# 30. SAVE CLUSTERED DATASET
# ============================================================

output_file = "hungarian_clustered.csv"

df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print(
    f"\nClustered dataset saved as: {output_file}"
)


# ============================================================
# 31. SAVE CLUSTER SUMMARY
# ============================================================

summary_file = "hungarian_cluster_summary.csv"

cluster_summary.to_csv(
    summary_file,
    index=False,
    encoding="utf-8-sig"
)

print(
    f"Cluster summary saved as: {summary_file}"
)


# ============================================================
# 32. FINAL RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("FINAL RESULTS")
print("=" * 70)

print(
    f"Original documents: {len(df)}"
)

print(
    f"TF-IDF features: {X.shape[1]}"
)

print(
    f"Optimal K: {best_k}"
)

print(
    f"Silhouette Score: {final_silhouette:.4f}"
)

print(
    "\nDocuments per cluster:"
)

for cluster_number in range(best_k):

    count = len(
        df[df["cluster"] == cluster_number]
    )

    print(
        f"Cluster {cluster_number}: {count} documents"
    )

print("\nTop keywords:")

for cluster_number in range(best_k):

    print(
        f"\nCluster {cluster_number}:"
    )

    print(
        ", ".join(
            cluster_keywords[
                cluster_number
            ][:10]
        )
    )

print("\nOutput files:")
print("-", output_file)
print("-", summary_file)

print("\nDone!")

Libraries imported successfully.
Dataset loaded successfully.
Shape: (341, 7)

Column names:
['id', 'recipeName', 'rating', 'totalTimeInSeconds', 'course', 'cuisine', 'ingredients']

First 5 rows:


,id,recipeName,rating,totalTimeInSeconds,course,cuisine,ingredients
0,Chicken-Paprikash-_-Dumplings-1353545,Chicken Paprikash & Dumplings,3,5100.0,[Main Dishes],[Hungarian],"[chicken, margarine, onions, salt, pepper, pap..."
1,Traditional-Hungarian-Goulash-_Gulyas_-1339556,Traditional Hungarian Goulash (Gulyas),4,3900.0,[],[Hungarian],"[lard, yellow onion, hungarian paprika, beef, ..."
2,Delicious-Crockpot-Hungarian-Goulash-1362359,Delicious Crockpot Hungarian Goulash,3,600.0,[Main Dishes],[Hungarian],"[vegetable oil, mushrooms, onions, red bell pe..."
3,Slow-Cooker-Hungarian-Goulash-1314072,Slow Cooker Hungarian Goulash,4,3900.0,[Main Dishes],[Hungarian],"[chuck roast, low sodium beef broth, yellow on..."
4,Hungarian-Cucumber-Salad_-_Uborkasalata_-_pale...,"Hungarian Cucumber Salad, ""Uborkasalata"" (paleo)",4,1200.0,[Salads],[Hungarian],"[english cucumber, coconut cream, apple cider ..."



DATASET INFORMATION

Number of rows: 341
Number of columns: 7

Data types:
id                        str
recipeName                str
rating                  int64
totalTimeInSeconds    float64
course                    str
cuisine                   str
ingredients               str
dtype: object

Missing values:
id                     0
recipeName             0
rating                 0
totalTimeInSeconds    15
course                21
cuisine                0
ingredients            0
dtype: int64

Duplicate rows: 0

Dataset after removing duplicates:
(341, 7)

Possible text columns:


ValueError: No text column was found in the CSV file.